# Python in the Smoke 💨 🐍 💨
## Notebook h: Cleaning data

---

**Repo:** [`python-in-the-smoke`](https://github.com/MarkGotham/python-in-the-smoke)

**By:** Mark Gotham

**Licence:** MIT

**Colour and symbol key:**
- 🟢 A block of information.
- 🟣 A core exercise.
- ⭐ An optional exercise.
- ✅ The solution to the exercise above it.
- 📝 A note with important information.
- ⚠️ Common errors, and how to fix them.
- `Grey text like this` shows an exact code or command. Type it exactly as shown, including any capital letters, spaces, and punctuation.

**Assumed knowledge:**

This notebook builds on notebooks a to g, and continues directly from Notebook g (files, folders, and `read_csv`).
Each idea in the table below has appeared before,
and now returns in the form that's directly relevant for cleaning data.

| a-g | Assumed knowledge      | How it appears here                                                              |
|-----|------------------------|----------------------------------------------------------------------------------|
| a   | text (strings)         | Tidying text: stray spaces, capital letters, symbols such as `£`                 |
| c   | Booleans               | Tests such as `.isna()` that give `True` or `False`                              |
| e   | arguments              | Named arguments such as `errors="coerce"` and `subset=[...]`                     |
| f   | dataframes             | A dataframe from a dict of lists; `.str` methods; filtering; `.sum()` on Booleans |
| g   | `read_csv`, `to_csv`   | Reading a messy file; saving the cleaned version under a new name                |

**Learning Objectives**: After completing this notebook, you will be able to ...
- Explain what a missing value is, and how it differs from zero or empty text
- Audit a table for missing values, inconsistent labels and repeated rows
- Standardise text without guessing missing information
- Convert messy numbers and dates, and inspect every failed conversion
- Save a cleaned table under a new name, with a record of what you changed

---
## 🟢 Information: What is "cleaning"?

Real data almost never arrives ready to use.
Notebook g gave a first glimpse:
missing boroughs, an ID stored as a decimal, and a date stored as text.

**Cleaning** means finding problems like these, then deciding what to do about each one.
It does *not* mean making the data look tidy at any cost.
Often the right decision is to leave a value missing, and to say so.

This notebook follows five habits:
1. **Look first.** Find problems by eye before using code.
2. **Count before you change.** Know how many values each problem affects.
3. **Work on a copy.** Keep the raw data exactly as it arrived.
4. **Never guess.** Don't fill in a value unless the data itself supports it.
5. **Keep a record.** Write down what you changed, why, and what you left unresolved.

📝 Note: the programme we clean first is **fictional**.
Event names, dates, prices and attendance figures are invented for practice.
The problems in it are deliberate, but typical of real data.
None of it is a claim about real events or boroughs.
At the end, we return to the real music venues from Notebook g.

---
## 🟢 Information: Making a messy data file

Normally, a data file arrives from somewhere else.
To keep this notebook self-contained, we'll make ours instead.
That's a good chance to practise skills from Notebooks f and g:
1. build a dataframe from a dict of lists (Notebook f),
2. save it with `to_csv` (Notebook g),
3. read it back with `read_csv` (Notebook g), as if it had arrived from elsewhere.

Imports and variables don't carry over between notebooks, so we import again.
`Path("data").mkdir(exist_ok=True)` makes sure the `data` folder exists (as in Notebook g).

In [ ]:
from pathlib import Path
import pandas as pd

Path("data").mkdir(exist_ok=True)

📝 Note: run the next cell, but **don't study it yet**.
Its job is to create the file.
The code contains every problem in plain sight,
and the exercise that follows is about spotting them *in the file*, the way you would with real data.

In [ ]:
programme = pd.DataFrame(
    {
        "event_id": [
            "C01", "C02", "C03", "C04", "C05", "C06", "C07", "C07",
            "C08", "C09", "C10", "C11", "C12", "C13", "C14", "C15",
        ],
        "event": [
            "Canal Sounds", "Paper Stories", "Southbank Sketches", "Night Moves",
            "Stage by the River", "Local Voices", "Brass in the Park", "Brass in the Park",
            "Print Club", "Moving Together", "Small Stage", "Poetry after Dark",
            "Summer Chorus", "Open Studio", "Dance on the Green", "Curtain Call",
        ],
        "borough": [
            " Hackney ", "camden", "Lambeth", "HACKNEY", "Southwark", None, "Camden", "Camden",
            "Islington", "Lambeth", "Southwark", "Islington", "Hackney", "Camden", "Lambeth", "Southwark",
        ],
        "artform": [
            "music", "literature", "visual art", "dance", "theatre", "literature", "music", "music",
            "visual art", "dance", "theatre", "literature", "music", "visual art", "dance", "theatre",
        ],
        "event_date": [
            "2026-06-06", "2026-06-07", "2026-06-08", "2026-06-09", "2026-06-10", "2026-06-11",
            "2026-06-12", "2026-06-12", "2026-06-13", "2026-06-14", "2026-06-15", "2026-06-16",
            "2026-06-17", "2026-06-18", "not recorded", "2026-06-20",
        ],
        "ticket_price": [
            "£12", "Free", "£8", "£15", "£18", "Free", "£10", "£10",
            "tbc", "£6", "£12", "£5", "Free", "£4", "£7", "£20",
        ],
        "capacity": [120, 60, 80, 100, 150, 50, 200, 200, 40, 90, 70, 55, 180, 65, 110, 130],
        "attendance": [98, 54, 72, 86, 125, 41, 160, 160, 32, None, 65, 49, 171, 52, 88, 104],
    }
)

Now save it as a CSV file, with no index (Notebook g):

In [ ]:
programme.to_csv("data/practice_data.csv", index=False)

---
## 🟢 Information: Load and look first

A CSV file is plain text (Notebook g).
So before loading anything into pandas, we can read the file as text and see exactly what is stored.
`.read_text()` reads a whole file as one piece of text.
This should look about the same as it does if you open the csv file in a plain text editor (give or take quotatation marks).

In [ ]:
print(Path("data/practice_data.csv").read_text())

Now load it as a table.
This is a small file, so we can display the whole thing.
(Real files rarely allow that: that's what `.head()` is for, as in Notebook f.)

In [ ]:
raw = pd.read_csv("data/practice_data.csv")
raw

In [ ]:
raw.shape

In [ ]:
raw.info()

Each row should describe one event.
- `event_id` identifies it, and `event` is its title.
- `borough` and `artform` are categories.
- `event_date` is the day it took place.
- `ticket_price` is the advertised price *per person*, not revenue.
- `capacity` is the number of places available, and `attendance` is the number of people who came.

📝 Note: look at the `Dtype` column of `.info()`.
`attendance` is `float64` (decimals), although people come in whole numbers.
You saw the same thing with `os_addressbase_uprn` in Notebook g:
a column of whole numbers becomes decimals if any value is missing.
(That's also why the text file above shows `98.0` rather than `98`.)
(Text columns show as `str`, or `object` on older versions of pandas: see Notebook f.)

---
### 🟣 Core Exercise: Eyeball it first

Before using any more code, look at the text and the table above.
Make a list of every problem you can see (there are at least eight).

Hints:
- labels that are the same thing, written differently,
- empty values,
- rows that are repeated,
- values that are not what their column should hold.

---
---
---

### ✅ Solution: Eyeball it first

1. `borough`: ` Hackney ` (with spaces), `HACKNEY` and `Hackney` are all the same borough.
2. `borough`: `camden` (lower case) and `Camden` are the same borough.
3. `borough`: empty for `Local Voices`.
4. Row `C07`, `Brass in the Park`, appears twice, identical.
5. `ticket_price`: stored as text, with a `£` symbol.
6. `ticket_price`: contains `Free` (text, but we can say what it means).
7. `ticket_price`: contains `tbc` ("to be confirmed": text, and *unknown*).
8. `attendance`: empty for `Moving Together`.
9. `event_date`: contains `not recorded` where a date should be.
10. `event_date`: holds dates as text, not as dates.

Some of these are easy to see but can't be found by a missing-value test, as the next section shows.

---
## 🟢 Information: What is a missing value?

A **missing value** marks a place in the table where we don't have an answer.
pandas uses special markers for it, and which one you see depends on the column:
- `NaN` stands for "Not a Number".
This appears in number columns (which makes sense)
and often in text columns too (which is perhaps less intuitive).
- `NaT` means "Not a Time" and serves date columns.
- `<NA>` may appear in some newer column types.

`.isna()` recognises all of these, and gives `True` where a value is missing.
(`.notna()` is the opposite.)

Missing does **not** mean zero, and it does not mean empty text.
- An event with an `attendance` of `0` is an event that nobody attended. (A sad story, at least for the organisers.)
- An event with a missing `attendance` is an event where we don't know. (Neutral.)

Those are different claims, and mixing them up quietly changes your results.
Here's a tiny example, with one missing value:

In [ ]:
numbers = pd.Series([10, None, 30])
numbers

In [ ]:
numbers.isna()

In [ ]:
numbers.mean()

The mean is 20: `(10 + 30) / 2`.
pandas skipped the missing value; it did not treat it as `0` (which would give `(10 + 0 + 30) / 3`, about 13.3).
Most pandas summaries skip missing values in this way.

⚠️ A warning: text such as `"tbc"` or `"not recorded"` is **not** a missing value as far as pandas is concerned.
It is just text, so `.isna()` will not count it.
That is one reason to look at the data first.

---
## 🟢 Information: Counting categories with `value_counts`

`.value_counts()` counts how many times each distinct value appears in a column.
By default, it leaves out missing values.
Add `dropna=False` to include them (shown as `NaN`).

In [ ]:
raw["artform"].value_counts()

---
### 🟣 Core Exercise: A first audit

1. Count the missing values in every column with `raw.isna().sum()`.
(This works because `True` counts as `1`: the `.sum()` trick from Notebook f.)
2. Count the borough labels, including missing ones, with `raw["borough"].value_counts(dropna=False)`.
3. Compare the results with your list from the last exercise:
which of your problems did these two commands find, and which did they miss?

---
---
---

### ✅ Solution: A first audit

In [ ]:
raw.isna().sum()

In [ ]:
raw["borough"].value_counts(dropna=False)

**Found:**
- one missing `borough` and one missing `attendance` (the `1`s in the first output),
- the borough variants: ` Hackney `, `HACKNEY` and `Hackney` are counted separately, and so are `camden` and `Camden`.

**Missed:**
- the repeated row `C07`,
- `Free` and `tbc` (and the `£` symbol) in `ticket_price`,
- `not recorded` in `event_date`: it's text, so it is not counted as missing.

Counting finds some problems, and looking finds others.
You need both.

---
## 🟢 Information: Work on a copy

Before changing anything, make a separate working version.
The raw table then stays exactly as it arrived, so we can compare before and after (and start again if we need to).

Be careful: creating a new object (let's call it `clean`) from the existing one (`raw`)
with the syntax `clean = raw` does **not** make a copy.
It gives the *same* table a second name.

Watch what happens in this small example:

In [ ]:
small = pd.DataFrame({"a": [1, 2]})
alias = small
alias["b"] = 3
small

We changed `alias`, but `small` changed too: there was only ever one table, with two names.

Now the same thing with `.copy()`:

In [ ]:
small = pd.DataFrame({"a": [1, 2]})
separate = small.copy()
separate["b"] = 3
small

As the name suggests, `.copy()` makes a genuinely separate table, so `small` is untouched.
This case is clear when you see the name `copy`;
the potential pitfall is forgetting to do that step, or using the `x = y` syntax above..

We do this same copy step for our data.
From here on, `raw` stays as it arrived and `clean` is the one we change.

In [ ]:
clean = raw.copy()

---
## 🟢 Information: Standardise text

Two more tools from the same `.str` family as `.str.upper()` (cf Notebook f) will be useful here:
- `.str.strip()` removes spaces at the beginning and end of each value.
- `.str.title()` puts each word in Title Case, so `HACKNEY` becomes `Hackney` and `camden` becomes `Camden`.

Like most pandas methods, they *return* a new column and leave the original alone.
To keep the result, store it back in the column.
Missing values stay missing.

📝 Note: be selective.
Title case suits one-word labels like our boroughs.
It does **not** suit identifiers, people's names or titles, where capitals can matter,
and it turns names like `City of London` into `City Of London`.

In [ ]:
clean["borough"] = clean["borough"].str.strip().str.title()
clean["borough"].value_counts(dropna=False)

---
### 🟣 Core Exercise: Find the unknown borough

Use `.isna()` as a filter (Notebook f) to display the event with no borough.
Show its `event_id`, `event` and `borough`.
Should we assign a borough from its title?

---
---
---

### ✅ Solution: Find the unknown borough

In [ ]:
clean[clean["borough"].isna()][["event_id", "event", "borough"]]

`Local Voices` has no recorded borough.

The title gives us no reliable evidence of what the borough might be,
so we leave the value missing and make a note of it.

For information:
- `.notna()` is the opposite test to `.isna()`.
- Use these in preference to equality tests such as `== pd.NA`, which don't work as you'd hope for finding missing values.

---
## 🟢 Information: Duplicates: count before removing

`.duplicated()` marks repeated rows as `True`.
The first time a row appears is not marked; only later repeats are.
Add `keep=False` to mark *every* member of a repeated set instead.

An exact repeated row can be removed here because this file is defined as one row per event.
In other datasets, identical rows can be separate things (two identical sales, say), so check what a row means first.

In [ ]:
clean.duplicated().sum()

In [ ]:
clean[clean.duplicated(keep=False)]

---
### 🟣 Core Exercise: Remove the repeated row

1. Store the result of `.drop_duplicates()` back in `clean`, and add `.copy()` on the end.
(On some versions of pandas, adding columns to a table made by dropping or selecting rows triggers a `SettingWithCopyWarning`.
A fresh copy avoids that.)
2. Check the shape of the result.
3. Check for repeated `event_id` values, using `.duplicated()` on that column.

---
---
---

### ✅ Solution: Remove the repeated row

In [ ]:
clean = clean.drop_duplicates().copy()
clean.shape

In [ ]:
clean["event_id"].duplicated().sum()

There are now 15 rows and 8 columns, and no repeated `event_id`.

📝 Note: the row labels (the index) keep their old numbers, so there's now a gap where the repeated row was.
That's fine, and it's useful: the labels still match the rows in `raw`.

If an ID had appeared twice with *different* details, removing exact duplicates would not tell us which record was right.
That needs a decision (and a note in the cleaning log), not a command.

---
## 🟢 Information: Text to numbers: prices

In our data, prices are text: `£12`, `Free` and `tbc`.
We can't calculate with text, so we need to convert to numbers.
First, tidy the text; then convert it:
- `.replace({"Free": "0"})` swaps *whole* values.
- `.str.replace("£", "", regex=False)` removes a character from *within* each value.
(`regex=False` means "treat `£` literally". The `regex` argument refers to "regular expressions" and that's a topic for another day.)
- `pd.to_numeric(..., errors="coerce")` converts what it can, and turns anything it can't convert into a missing value.

This last `coerce` option is convenient, but watch out: it fails *silently* without drawing your attention to the error.
I.e., if there's a typo, it becomes a missing value without any error message.
So after converting, always check what failed.
We also keep the original `ticket_price` column, and put the numbers in a new column: `price_gbp`.

In [ ]:
price_text = clean["ticket_price"].str.strip().replace({"Free": "0"})
price_text = price_text.str.replace("£", "", regex=False)
clean["price_gbp"] = pd.to_numeric(price_text, errors="coerce")
clean[["event", "ticket_price", "price_gbp"]]

---
### 🟣 Core Exercise: Check the conversion

1. Count the missing values before and after conversion:
compare `clean["ticket_price"].isna().sum()` with `clean["price_gbp"].isna().sum()`.
2. Display the events with a missing `price_gbp`.
3. Would it be right to fill that missing price with zero?

---
---
---

### ✅ Solution: Check the conversion

In [ ]:
print(clean["ticket_price"].isna().sum())
print(clean["price_gbp"].isna().sum())

In [ ]:
clean[clean["price_gbp"].isna()][["event", "ticket_price", "price_gbp"]]

Before: 0 missing.
After: 1 missing, and it's the one we expected: `Print Club` has `tbc`.
(If the count had been bigger than expected, we'd have found a conversion that failed unexpectedly.)

Is it right to fill with zero? **No!**
- `tbc` means the price is unknown.
- `0` would mean *free*, which is a different claim.
- So we leave it as a missing value (`NaN`).

---
## 🟢 Information: Text to dates: say which format you expect

`pd.to_datetime` converts date text into proper dates.
To do that safely, tell it the format with `format=`, using these codes:

| Code | Meaning             | Example in `2026-06-06` |
|------|---------------------|-------------------------|
| `%Y` | year (four digits)  | `2026`                  |
| `%m` | month (two digits)  | `06`                    |
| `%d` | day (two digits)    | `06`                    |

Our file uses year-month-day, so the format is `"%Y-%m-%d"`.
(Always say which. `06/07/2026` is 6 July in the UK, and 7 June in the US.)

As with prices, `errors="coerce"` turns anything that doesn't fit into a missing value.
For dates, the marker is `NaT` ("Not a Time").
We use a new column (`date`), so the original text in `event_date` is still available for checking.

In [ ]:
clean["date"] = pd.to_datetime(clean["event_date"], format="%Y-%m-%d", errors="coerce")
clean[["event", "event_date", "date"]]

---
### 🟣 Core Exercise: Check the dates

1. Display the events where `date` is missing.
2. Is that the number of failures you expected, given your eyeball list?
3. Should the rest of that event's record be thrown away?

---
---
---

### ✅ Solution: Check the dates

In [ ]:
clean[clean["date"].isna()][["event", "event_date", "date"]]

One event: `Dance on the Green`, whose `event_date` is `not recorded`.
That matches our eyeball list: exactly one date was unusable.

Should we throw the rest of the record away? No.
A missing date doesn't make its capacity, price or attendance useless.
We keep the row, and just don't use `date` for that event when the date matters.

---
### ⭐ Optional Exercise: Do the values make sense?

Missing is not the only kind of problem: some values are simply implausible.
Count, using comparisons and `.sum()` (as in Notebook f):
1. known prices that are negative,
2. capacities that are zero or negative,
3. events where `attendance` is bigger than `capacity`.

---
---
---

### ✅ Solution: Do the values make sense?

In [ ]:
(clean["price_gbp"] < 0).sum()

In [ ]:
(clean["capacity"] <= 0).sum()

In [ ]:
(clean["attendance"] > clean["capacity"]).sum()

All three counts are zero.
But remember exactly what the checks can and can't see;
the comparisons above return `False` for the `NaN` values in our numeric columns.
They therefore do not detect missing values; check those separately with `.isna()`.

comparison involving a missing value gives `False` (or similar), not `True`.
So an event with missing `attendance` is *not* flagged by the third check.
Missing values have to be checked for separately (as we did earlier).

---
## 🟢 Information: Missing values: choose for the question

What should we do with the missing values that remain?
Two tools are available:
- `.dropna(subset=["attendance"])` removes rows where `attendance` is missing, and leaves other missing fields alone.
- `.fillna(...)` supplies a replacement value.

Neither is always the right choice.
Our approach:
- Keep all 15 events in the main table (`clean`).
- For each question, select just the rows with the values *that question* needs.
- If you need a label for a missing category in a count, use `"Not recorded"` as a *display label* (see Notebook i), not as a change to the data.

---
### 🟣 Core Exercise: A table for attendance only

Create a new dataframe called `attendance_known`, dropping rows only when `attendance` is missing.
Display its shape.
Do **not** overwrite `clean`.

---
---
---

### ✅ Solution: A table for attendance only

In [ ]:
attendance_known = clean.dropna(subset=["attendance"])
attendance_known.shape

14 rows remain (and 10 columns, now that we've added `price_gbp` and `date`).
Dropping every row with *any* missing field would also lose events with a missing borough, price or date,
even where their attendance is known.

---
## 🟢 Information: Save and document

Now save the cleaned table.
Following Notebook g, we use a **new name** and no index, and leave the original file untouched:

In [ ]:
clean.to_csv("data/practice_data_clean.csv", index=False)

Notebooks i and j will load this file, so let's check that it reads back properly.
A CSV stores everything as text, so dates come back as plain text unless we say otherwise:

In [ ]:
check = pd.read_csv("data/practice_data_clean.csv")
check.dtypes

`date` is text again (`str` or `object`).
`parse_dates=["date"]` tells `read_csv` which columns hold dates:

In [ ]:
check = pd.read_csv("data/practice_data_clean.csv", parse_dates=["date"])
print(check.shape)
check.dtypes

Now `date` is a proper date column (`datetime64`), and the shape is 15 rows by 10 columns.
Notebooks i and j load the file in exactly this way.

---
### 🟣 Core Exercise: Your cleaning log

Replace this text with your cleaning log: a record of what you changed, why, and what you left unresolved.
One short sentence per decision is enough.
Include:
- every change you made,
- every problem you left unresolved (and why),
- the fact that this data is fictional.

Double-click this cell to edit it.

---
---
---

### ✅ Solution: Your cleaning log

**Example cleaning log**

- Standardised `borough` (spaces and capitals): `Hackney` and `Camden` now each have a single spelling.
- Removed one exact repeated row (`C07`), after checking that `event_id` is then unique.
- Converted prices to a new numeric column, `price_gbp`: `Free` is £0, `tbc` is left unknown.
- Parsed dates into a new column, `date`: `not recorded` is left unknown.
- Left unresolved: one missing `borough` (`Local Voices`), one missing `attendance` (`Moving Together`), one unknown price and one unknown date.
- Kept the original text columns, and the original file (`practice_data.csv`), unchanged.
- All records are fictional teaching data.

---
## 🟢 Information: Back to the real venues

In Notebook g, we loaded a real file, `Music_venues_grassroots.csv`, and noticed some surprises.
Now we have the tools to deal with them.
(If you don't have the file, go back to the core exercise in Notebook g that downloads it.)

| Surprise in Notebook g               | What's going on                                                                                                                    | What we do                                           |
|--------------------------------------|------------------------------------------------------------------------------------------------------------------------------------|------------------------------------------------------|
| `borough_name` has 14 missing values | Unrecorded. A venue's name can hint at a borough (e.g., `Oslo Hackney`), but a name is not evidence.                               | Leave them missing, and count them                   |
| `HACKNEY` in capitals, once          | Inconsistent capitals                                                                                                              | Standardise (core exercise below)                    |
| `os_addressbase_uprn` is `float64`   | An ID (a label, not a measurement), stored as decimals because some values are missing; and some of the values may be placeholders | Read it as text, and look closer (optional exercise) |
| `runtime` is text                    | A date, month first (`11/27/2022`), and the same in every row: it says nothing about the venues                                    | Convert it (optional exercise)                       |
| `address3` is almost empty           | Only 3 of 100 venues have a value                                                                                                  | Don't rely on it: nothing to repair                  |

---
### 🟣 Core Exercise: Standardise the real boroughs

1. Load the file as `music_raw`, and make a copy called `music_clean`.
2. Standardise `borough_name` in `music_clean` with `.str.strip().str.title()`.
3. Count the venues whose `borough_name` is exactly `"Hackney"` before (`music_raw`) and after (`music_clean`).
4. Count the venues with no `borough_name`.
5. Compare `.value_counts(dropna=False)` before and after: what changed?

---
---
---

### ✅ Solution: Standardise the real boroughs

In [ ]:
music_raw = pd.read_csv("data/Music_venues_grassroots.csv")
music_clean = music_raw.copy()
music_clean["borough_name"] = music_clean["borough_name"].str.strip().str.title()

print((music_raw["borough_name"] == "Hackney").sum())
print((music_clean["borough_name"] == "Hackney").sum())
print(music_clean["borough_name"].isna().sum())

In [ ]:
music_raw["borough_name"].value_counts(dropna=False)

In [ ]:
music_clean["borough_name"].value_counts(dropna=False)

- Hackney goes from 14 to 15: the venue written as `HACKNEY` (`Moth Club`, in the very first row) has joined the others.
- 14 venues have no borough, and we've left them that way.
- In the counts, that is the only change: `HACKNEY` has merged into `Hackney`, and nothing else moved.
Always check what a standardising step *did*.
(This file already writes `City Of Westminster` in title case, so title-casing leaves it alone.
Other sources write `Westminster` or `City of Westminster`: matching names *between* files is a cleaning job in its own right.)

Is 15 the true number of venues in Hackney? **Not necessarily.**
Some of the 14 venues with no borough might be in Hackney.
The count tells us how many venues are *recorded as* Hackney, which is not the same thing.

---
### ⭐ Optional Exercise: A closer look at the IDs and dates

1. Read the file again, this time as `music_ids`, telling pandas to read `os_addressbase_uprn` as text:
`pd.read_csv("data/Music_venues_grassroots.csv", dtype={"os_addressbase_uprn": "string"})`.
Display the `name` and `os_addressbase_uprn` columns of both versions (`music_raw` and `music_ids`), using `.head()`.
2. A UPRN (Unique Property Reference Number) is meant to identify one property, so IDs should rarely repeat.
Check with `music_ids["os_addressbase_uprn"].value_counts().head(3)`.
What do you notice?
3. In `music_ids`, convert `runtime` into a new column called `runtime_date`, using the format `"%m/%d/%Y"` (month first).
How many different values does `runtime_date` have, and what does that tell you?

---
---
---

### ✅ Solution: A closer look at the IDs and dates

In [ ]:
music_ids = pd.read_csv("data/Music_venues_grassroots.csv", dtype={"os_addressbase_uprn": "string"})
print(music_raw[["name", "os_addressbase_uprn"]].head())
print(music_ids[["name", "os_addressbase_uprn"]].head())

In [ ]:
music_ids["os_addressbase_uprn"].value_counts().head(3)

In [ ]:
music_ids["runtime_date"] = pd.to_datetime(music_ids["runtime"], format="%m/%d/%Y")
music_ids["runtime_date"].nunique()

**1. Text keeps IDs as written.**
An ID is a label, not a measurement: you'd never average it or add two together, so text is the right type.
Read as text, it keeps its digits as written: no scientific notation (`1.000230e+11`), and no trailing `.0`.
(The `dtype` argument says what type a column should have.
Missing values show as `<NA>` in this text type: yet another spelling of "missing".)

**2. Placeholders.**
The value `100000000000` appears 25 times, and `0` appears 5 times.
These look suspiciously like **placeholders**:
values typed in because *something* had to go in the field.
Thirty rows contain these two suspicious values, 16 have missing IDs, and 54 contain other non-missing values.
These counts alone do not establish with certainty which IDs are valid,
but they do prompt us to conduct additional checks of these values specifically and the source documentation.
Placeholders are really missing values in disguise, and `.isna()` can't see them.
We don't fix this by guessing:
we avoid using this column to identify venues, and we record the problem in the cleaning log.
(Converting the placeholders to missing values would also be defensible, as long as you say so.)

**3. A column with one value.**
`runtime_date` has just **1** distinct value.
A column that is identical in every row can't tell one venue from another,
so it is of limited use for analysis, even though it's now a "proper" date.
(It might be of some use in auditing the provenance of data).

---
## 🟢 Information: Summary

| To ...                          | Write ...                                                    |
|---------------------------------|--------------------------------------------------------------|
| Read a file as plain text       | `Path("data/file.csv").read_text()`                          |
| Make a separate working copy    | `clean = raw.copy()`                                         |
| Count missing values            | `df.isna().sum()`                                            |
| Count categories (with missing) | `df["col"].value_counts(dropna=False)`                       |
| Tidy category text              | `df["col"].str.strip().str.title()`                          |
| Find repeats                    | `df.duplicated()`                                            |
| Remove exact repeats            | `df.drop_duplicates().copy()`                                |
| Convert to numbers              | `pd.to_numeric(text, errors="coerce")`                       |
| Convert to dates                | `pd.to_datetime(text, format="%Y-%m-%d", errors="coerce")`   |
| Keep rows with a known value    | `df.dropna(subset=["col"])`                                  |
| Save under a new name           | `df.to_csv("data/new_name.csv", index=False)`                |
| Read dates back from a CSV      | `pd.read_csv("data/file.csv", parse_dates=["date"])`         |
| Read an ID as text              | `pd.read_csv("data/file.csv", dtype={"id_col": "string"})`   |

Remember: look first, count before you change, work on a copy, never guess, and keep a record.

---
## ⚠️ Common Errors

Most of the mistakes in cleaning are "silent": there's no error message, just a wrong or misleading result.

**1. Forgetting to store the result**

Most pandas methods *return* a result, and don't change your table.
`clean["borough"].str.strip()` on its own shows tidy values, but `clean` is unchanged.
Fix: store it, with `clean["borough"] = clean["borough"].str.strip()`.
The same applies to `.drop_duplicates()` and `.dropna()`.

**2. `clean = raw` is not a copy**

It gives the same table two names, so changes to `clean` also change `raw`.
Fix: `clean = raw.copy()`.

**3. Filling missing values with zero**

Unknown attendance is not zero attendance, and an unknown price is not a free event.
Fix: leave it missing, and select the rows you need for each question.

**4. Using `errors="coerce"` and never checking**

Anything that fails to convert becomes missing without a message.
Fix: count the missing values before and after, and display the new ones.

**5. Trusting a test that can't see missing values**

These comparisons return `False` for the `NaN` values in our columns,
so `attendance > capacity` doesn't flag a missing attendance.
Fix: check for missing values separately, using `.isna()` to check missingness explicitly.

**6. Title-casing everything**

`.str.title()` is right for simple labels, but not for identifiers, names or titles where capitals matter.
Fix: standardise only where you've checked that it's safe, and compare before and after.

**7. `FileNotFoundError` for the music venues**

The real file comes from Notebook g.
Fix: do the download exercise in Notebook g first, and check that the file is in `data`.

**8. `SettingWithCopyWarning`**

A warning (not an error) when you add a column to a table that pandas thinks is a view of another.
Fix: add `.copy()` when you create the working table, as in this notebook.

**9. Re-running cells out of order**

If the table no longer matches what you expect, use **Restart & Run All**.
This notebook is safe to re-run from the top: the first cells re-create the raw file each time.

End

---